# 06 — BioCLIP 2 (`hf-hub:imageomics/bioclip-2`) T0 Zero-Shot & T1 Cached Embedding Linear Probe

Follows the Milestone 9 training efficiency policy (§0.1 & §7):
1. **T0 Zero-Shot:** Evaluates `hf-hub:imageomics/bioclip-2` (ViT-L/14, MIT) on validation crops using 3 prompt templates per species.
2. **T1 Frozen Embeddings + Linear Head (Default Production Path):** Computes L2-normalized float16 embeddings once (4 views per train crop, 1 view per eval crop), caches them to `gs://$GCS_BUCKET/ucsc-wildlife/embeddings/bioclip2/crop-v1/`, and fits a group-stratified multinomial logistic regression head (`GroupKFold` on `observation_id`).

In [ ]:
!pip install -q open_clip_torch torch torchvision scikit-learn pyarrow pandas
import open_clip
import torch

assert torch.cuda.is_available(), "Select a Colab GPU runtime for BioCLIP 2 embedding extraction"
model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms("hf-hub:imageomics/bioclip-2")
tokenizer = open_clip.get_tokenizer("hf-hub:imageomics/bioclip-2")
model = model.cuda().eval()
print("Loaded BioCLIP 2 ViT-L/14 on GPU")

In [ ]:
!PYTHONPATH=src python3 -m wildlife.train \
  --config configs/train_v2.yaml \
  --manifest data/manifests/manifest_v2.parquet \
  --splits data/splits/split_v2.csv \
  --tier t1 \
  --require-torch \
  --output runs/classifier_v2_t1